# 02 – FDG PET/CT Segmentation Practice — Solution

**SNMMI AI Certificate – Hands-On Module**

This is the **complete solution** to the FDG practice notebook.
All three tasks are implemented with working code.

> **Run cells top-to-bottom on a fresh Colab runtime.**

---

**Dataset citation:**

> Gatidis, S., Kuestner, T., Ingrisch, M., Hepp, T., Frueh, M., Nikolaou, K.,
> La Fougere, C., Pfannenberg, C., Fabritius, M., Jeblick, K., Schachtner, B.,
> Dexl, J., Wesp, P., Mittermeier, A., Unterrainer, L., Sheikh, G., Boening, G.,
> Brendel, M., Ricke, J., … Cyran, C. (2025).
> *PSMA-FDG-PET-CT-Lesions*.
> University of Tuebingen, Ludwig-Maximilians-University Munich.
> [https://doi.org/10.57754/FDAT.0zs4c-89f12](https://doi.org/10.57754/FDAT.0zs4c-89f12)
>
> License: **CC BY-NC 4.0** – non-commercial use only.

## 0 · Setup & Config

In [ ]:
%pip install -q numpy matplotlib tqdm scikit-learn

import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

EPOCHS     = 3
BATCH_SIZE = 16
LR         = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {DEVICE}")

## 1 · Download & Unzip FDG Sample Data

In [ ]:
# ── Paste your GitHub Release URL for the FDG zip here ───────────────────────
DATA_URL = "https://github.com/ZekunLi-Zeke/snmmi-ai-hands-on-segmentation/releases/download/v0.1/fdg_sample.zip"
# ─────────────────────────────────────────────────────────────────────────────

import pathlib, urllib.request, zipfile

DATA_DIR = pathlib.Path("./data/fdg")
DATA_DIR.mkdir(parents=True, exist_ok=True)
zip_path = pathlib.Path("/tmp/fdg_sample.zip")

if not any(DATA_DIR.rglob("*.npz")):
    if not zip_path.exists():
        print("Downloading FDG sample data …")
        urllib.request.urlretrieve(DATA_URL, zip_path)
        print("Download complete.")
    print("Unzipping …")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_DIR)
    print("Done.")
else:
    print("Data already present.")

npz_files = sorted(DATA_DIR.rglob("*.npz"))
assert len(npz_files) > 0, "No .npz file found – check DATA_URL."
print(f"Found: {[f.name for f in npz_files]}")
data = np.load(npz_files[0])
for k in data:
    print(f"  {k}: shape={data[k].shape}  dtype={data[k].dtype}")

## 2 · Load Data & Visualise (Solution: augmentation ON)

In [ ]:
# Task 2 solution: AUGMENT = True (horizontal flip enabled)
AUGMENT = True

class PatchDataset(Dataset):
    """2-D PET/CT patch dataset from a .npz file."""

    def __init__(self, npz_path, augment=False):
        data = np.load(npz_path)
        self.pet    = data["pet"].astype(np.float32)
        self.ct     = data["ct"].astype(np.float32)
        self.mask   = data["mask"].astype(np.float32)
        self.augment = augment

    def __len__(self):
        return len(self.pet)

    def __getitem__(self, idx):
        pet  = self.pet[idx].copy()
        ct   = self.ct[idx].copy()
        mask = self.mask[idx].copy()

        if self.augment and random.random() > 0.5:
            # Task 2: horizontal flip augmentation
            pet  = pet[:, ::-1].copy()
            ct   = ct[:, ::-1].copy()
            mask = mask[:, ::-1].copy()

        x = np.stack([pet, ct], axis=0)
        y = mask[np.newaxis]
        return torch.from_numpy(x), torch.from_numpy(y)


npz_path = sorted(DATA_DIR.rglob("*.npz"))[0]
full_ds  = PatchDataset(npz_path, augment=AUGMENT)
n_total  = len(full_ds)
n_val    = max(1, int(0.2 * n_total))
n_train  = n_total - n_val

train_ds, val_ds = random_split(
    full_ds, [n_train, n_val],
    generator=torch.Generator().manual_seed(SEED)
)
print(f"Dataset: {n_total} patches  |  train {n_train}  val {n_val}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  drop_last=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for col in range(4):
    x, y = full_ds[col]
    axes[0, col].imshow(x[0].numpy(), cmap="hot");   axes[0, col].set_title(f"PET #{col}")
    axes[1, col].imshow(x[1].numpy(), cmap="gray");  axes[1, col].set_title(f"CT #{col}")
    axes[2, col].imshow(y[0].numpy(), cmap="gray");  axes[2, col].set_title(f"Mask #{col}")
for ax in axes.flat:
    ax.axis("off")
plt.suptitle("Sample patches (PET / CT / Mask)", fontsize=13)
plt.tight_layout()
plt.show()

## 3 · Define Compact 2-D U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.block(x)


class UNet2D(nn.Module):
    def __init__(self, in_ch=2, out_ch=1, features=16):
        super().__init__()
        f = features
        self.enc1 = DoubleConv(in_ch, f);  self.enc2 = DoubleConv(f, f*2)
        self.enc3 = DoubleConv(f*2, f*4);  self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(f*4, f*8)
        self.up3  = nn.ConvTranspose2d(f*8, f*4, 2, stride=2); self.dec3 = DoubleConv(f*8, f*4)
        self.up2  = nn.ConvTranspose2d(f*4, f*2, 2, stride=2); self.dec2 = DoubleConv(f*4, f*2)
        self.up1  = nn.ConvTranspose2d(f*2, f,   2, stride=2); self.dec1 = DoubleConv(f*2, f)
        self.head = nn.Conv2d(f, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x); e2 = self.enc2(self.pool(e1)); e3 = self.enc3(self.pool(e2))
        b  = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], 1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], 1))
        return self.head(d1)


model    = UNet2D(in_ch=2, out_ch=1, features=16).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters: {n_params:,}")

## 4 · Define Loss & Optimizer (Solution: Dice + BCE)

In [ ]:
# Task 1 solution: Dice + BCE combined loss
def dice_loss(logits, target, smooth=1.0):
    pred  = torch.sigmoid(logits)
    inter = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return 1.0 - ((2.0 * inter + smooth) / (denom + smooth)).mean()


def loss_fn(logits, target):
    """Task 1 solution: Dice loss + Binary Cross-Entropy."""
    bce = nn.functional.binary_cross_entropy_with_logits(logits, target)
    return dice_loss(logits, target) + bce


optimizer = optim.Adam(model.parameters(), lr=LR)
print("Loss: Dice + BCE  |  Optimizer: Adam  |  LR:", LR)

## 5 · Training Loop

In [ ]:
def dice_score(logits, target, threshold=0.5):
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum(dim=(2, 3))
    denom = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    return ((2.0 * inter + 1.0) / (denom + 1.0)).mean().item()


history = {"train_loss": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [train]", leave=False):
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= n_train

    model.eval()
    val_dice_sum = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            val_dice_sum += dice_score(model(x), y) * x.size(0)
    val_dice = val_dice_sum / n_val

    history["train_loss"].append(epoch_loss)
    history["val_dice"].append(val_dice)
    print(f"Epoch {epoch:>2}: train loss={epoch_loss:.4f}  val Dice={val_dice:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(range(1, EPOCHS+1), history["train_loss"], marker="o")
ax1.set(title="Train Loss", xlabel="Epoch", ylabel="Loss")
ax2.plot(range(1, EPOCHS+1), history["val_dice"], marker="o", color="orange")
ax2.set(title="Validation Dice", xlabel="Epoch", ylabel="Dice")
plt.tight_layout(); plt.show()

## 6 · Visualise Predictions

In [ ]:
model.eval()
n_show = min(4, len(val_ds))
fig, axes = plt.subplots(n_show, 4, figsize=(14, 3.5 * n_show))

with torch.no_grad():
    for row in range(n_show):
        x, y  = val_ds[row]
        logit = model(x.unsqueeze(0).to(DEVICE))
        pred  = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()
        axes[row, 0].imshow(x[0].numpy(), cmap="hot");  axes[row, 0].set_title("PET")
        axes[row, 1].imshow(x[1].numpy(), cmap="gray"); axes[row, 1].set_title("CT")
        axes[row, 2].imshow(y[0].numpy(), cmap="gray"); axes[row, 2].set_title("GT Mask")
        axes[row, 3].imshow(pred,          cmap="gray"); axes[row, 3].set_title("DL Prediction")

for ax in axes.flat: ax.axis("off")
plt.suptitle("Predictions vs Ground Truth", fontsize=14)
plt.tight_layout(); plt.show()

## 7 · Task 3 Solution – Dice & Total Tumor Area

In [ ]:
def threshold_segment(pet_patch, fraction=0.4):
    """Threshold segmentation at *fraction* × per-patch max SUV."""
    max_suv = pet_patch.max()
    if max_suv == 0:
        return np.zeros_like(pet_patch)
    return (pet_patch > fraction * max_suv).astype(np.float32)


def arr_dice(a, b, smooth=1.0):
    inter = (a * b).sum()
    return (2.0 * inter + smooth) / (a.sum() + b.sum() + smooth)


def arr_iou(a, b, smooth=1.0):
    inter = (a * b).sum()
    union = ((a + b) > 0).sum()
    return (inter + smooth) / (union + smooth)


# ── Task 3 solution ───────────────────────────────────────────────────────────
model.eval()
all_dice, all_iou, gt_areas, dl_areas, thr_dices = [], [], [], [], []

with torch.no_grad():
    for i in range(len(val_ds)):
        x, y    = val_ds[i]
        pet_np  = x[0].numpy()
        gt_mask = y[0].numpy()

        logit   = model(x.unsqueeze(0).to(DEVICE))
        dl_pred = (torch.sigmoid(logit) > 0.5).float().cpu().squeeze().numpy()
        thr_mask = threshold_segment(pet_np)

        # Task 3a: Dice
        all_dice.append(arr_dice(dl_pred, gt_mask))
        # Task 3b: IoU
        all_iou.append(arr_iou(dl_pred, gt_mask))
        # Task 3c: tumour area
        gt_areas.append(int(gt_mask.sum()))
        dl_areas.append(int(dl_pred.sum()))
        # Bonus: threshold baseline Dice
        thr_dices.append(arr_dice(thr_mask, gt_mask))

# Task 3d: print summary
print(f"Mean Dice (DL):       {np.mean(all_dice):.3f}")
print(f"Mean IoU  (DL):       {np.mean(all_iou):.3f}")
print(f"Mean Dice (40% thr):  {np.mean(thr_dices):.3f}")
print(f"Mean GT tumour area:  {np.mean(gt_areas):.1f} px")
print(f"Mean DL tumour area:  {np.mean(dl_areas):.1f} px")
print(f"Total GT tumor area: {sum(gt_areas)} px")
print(f"Total DL tumor area: {sum(dl_areas)} px")

# Visualise DL vs threshold for one sample
i = 0
x, y     = val_ds[i]
pet_np   = x[0].numpy()
gt_mask  = y[0].numpy()
thr_mask = threshold_segment(pet_np)
with torch.no_grad():
    dl_pred = (torch.sigmoid(model(x.unsqueeze(0).to(DEVICE))) > 0.5).float().cpu().squeeze().numpy()

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
axes[0].imshow(pet_np,  cmap="hot");  axes[0].set_title("PET input")
axes[1].imshow(gt_mask, cmap="gray"); axes[1].set_title("GT Mask")
axes[2].imshow(dl_pred, cmap="gray"); axes[2].set_title(f"DL  (Dice={all_dice[i]:.2f})")
axes[3].imshow(thr_mask,cmap="gray"); axes[3].set_title(f"40% Thr (Dice={thr_dices[i]:.2f})")
for ax in axes: ax.axis("off")
plt.suptitle("DL Model vs 40 % SUV Threshold Baseline", fontsize=13)
plt.tight_layout(); plt.show()